# 06 Start vLLM with LoRA Enabled

This notebook documents the local vLLM launch path.

```mermaid
flowchart LR
    A[Qwen2.5-7B-Instruct] --> B[vLLM OpenAI server]
    C[adapters/*] --> B
    B --> D[/v1/chat/completions]
```

Dynamic adapter loading requires `VLLM_ALLOW_RUNTIME_LORA_UPDATING=True`.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Compose command

Run this in a terminal from the project root. Expected output: vLLM logs with an OpenAI API server on port 8000.

In [ ]:
print("make serve")

## Equivalent vLLM command

Use this on an MLIS host or Linux CUDA machine with vLLM installed.

In [ ]:
command = f'''
VLLM_ALLOW_RUNTIME_LORA_UPDATING=True \
python -m vllm.entrypoints.openai.api_server \
  --host ${'{'}VLLM_HOST:-0.0.0.0{'}'} \
  --port ${'{'}VLLM_PORT:-8000{'}'} \
  --model "{settings_cfg.base_model}" \
  --served-model-name base \
  --enable-lora \
  --max-model-len 4096
'''
print(command)

## Health check

Expected output: JSON model list containing `base` before adapters are loaded.

In [ ]:
import requests

response = requests.get(
    f"{settings_cfg.vllm_base_url}/v1/models",
    headers={"Authorization": f"Bearer {settings_cfg.vllm_api_key}"},
    timeout=10,
)
print(response.status_code)
print(response.text[:1000])